In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
folder = r'C:\Users\polti\OneDrive\Documents\FME\Trading\data'

df = pd.read_parquet(f'{folder}\\JPM.parquet')
print(df.columns)

df.rename(columns={'volumen': 'Volume'}, inplace=True)

# Agregar columna 'Adj Close' si no existe
if 'Adj Close' not in df.columns:
    df['Adj Close'] = pd.NA

# Lista de columnas en el orden deseado
ordered_cols = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'RVT',
    'positivePartscr', 'negativePartscr', 'fearPartscr', 'findownPartscr',
    'finupPartscr', 'finhypePartscr', 'certaintyPartscr', 'uncertaintyPartscr'
]

# Agregar columnas extra (las que no están en ordered_cols) al final
extra_cols = [col for col in df.columns if col not in ordered_cols]
final_cols = ordered_cols + extra_cols

# Reordenar el DataFrame
df = df[final_cols]
# Crear columna de índice numérico
df.insert(0, 'Index', range(len(df)))

# Guardar como nuevo parquet (opcional)
df.to_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\JPM_reord.parquet', index=False)

In [ ]:
import pandas_datareader.data as web, pandas as pd, os

hist = (web.DataReader('AAPL', 'stooq', '2009-01-02', '2012-03-01')
          .rename(columns=str.capitalize)
          .iloc[::-1]                 # Stooq viene descendente
          .reset_index()
          [['Date','Open','High','Low','Close']])
hist['Date'] = pd.to_datetime(hist['Date']).dt.date

out = r"C:\Users\polti\Downloads\lexicons\split\AAPL_price.parquet"
os.makedirs(os.path.dirname(out), exist_ok=True)
hist.to_parquet(out, index=False)
print("Descargado desde Stooq:", hist.shape[0], "filas")


ModuleNotFoundError: No module named 'pandas_datareader'

In [6]:
import yfinance as yf
import pandas as pd
import os

# 1️⃣ Descargar toda la serie diaria de AAPL (2009‑01‑02 → 2025‑05‑28)
hist = yf.download(
    'AAPL',
    start='2009-01-02',
    end  ='2025-05-28',
    interval='1d',
    threads=False,     # ❗ evita múltiplos hilos internos
    progress=False     # ❗ evita llamadas extra para la barra
).reset_index()

# 2️⃣ Seleccionar columnas
cols = ['Date', 'Open', 'High', 'Low', 'Close']
hist = hist[cols].copy()

# 3️⃣ Formato AAAA‑MM‑DD
hist['Date'] = pd.to_datetime(hist['Date']).dt.date

# 4️⃣ Guardar en Parquet
folder = r'C:\Users\polti\Downloads\lexicons\split'
os.makedirs(folder, exist_ok=True)
hist.to_parquet(os.path.join(folder, 'AAPL_price.parquet'), index=False)



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


In [ ]:
import pandas as pd
df1 = pd.read_parquet(r'C:\Users\polti\Downloads\lexicons\split\lexicons_HSBC.parquet')
df2 = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_price.parquet')

# Asegurarse de que la columna 'Date' esté en formato datetime
df1['Date'] = pd.to_datetime(df1['datetime'])
df2['Date'] = pd.to_datetime(df2['Date'])

# Usar 'Date' como índice
df1.set_index('Date', inplace=True)
df2.set_index('Date', inplace=True)

# Hacer el inner join (solo fechas que aparecen en ambos)
merged = df2.join(df1, how='inner', lsuffix='_df1', rsuffix='_df2')
merged['Date'] = pd.to_datetime(merged.index).date
# Guardar el resultado
merged.to_parquet(r'C:\Users\polti\Downloads\lexicons\split\HSBC_trans.parquet', index=False)
print(merged.columns)

Index(['Open', 'High', 'Low', 'Close', 'datetime', 'UpDownLogRatio',
       'sentiment_score', 'Date'],
      dtype='object')


In [ ]:
print(merged.columns)

In [ ]:
import pandas as pd
#abrimos el parquet de DB_reord
df = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.parquet')

df.drop(df.columns[0], axis=1, inplace=True)

# Establir la columna "Date" (o la corresponent) com a índex i convertir a datetime
df.set_index(df.columns[0], inplace=True)
df.index = pd.to_datetime(df.index)

# Crear les columnes de BULL, BEAR i finalment la BBr
df["BULL"] = 0.33 * (df["positivePartscr"] + df["certaintyPartscr"] + df["finhypePartscr"])
df["BEAR"] = 0.33 * (df["negativePartscr"] + df["uncertaintyPartscr"] + df["findownPartscr"])
df["BBr"] = 100 * df["BULL"] / (df["BULL"] + df["BEAR"])


#ver los valores de BBr de HSBC desde el 2023 al 2025
#ver tambien cuantos valores de BBr son superiores a 70
print(df.loc['2023-01-01':'2025-03-22', ['BBr']].head(10))
print("Valores de BBr superiores a 70:", (df['BBr'] > 74).sum())


: 

In [ ]:
df_70 = df.iloc[:int(len(df) * 0.7)]
print(df_70["BBr"].mean())
print(df_70["BBr"].min())
print(df_70["BBr"].max())
print(df_70["BBr"].std())
print(df_70["BBr"].median())
print(df_70["BBr"].quantile(0.25))
print(df_70["BBr"].quantile(0.75))

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.parquet')

#transformamos df a csv y guardamos en la misma carpeta que el parquet
df.to_csv(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.csv', index=False)

In [15]:
import pandas as pd

# Ruta del archivo original
file_path = r'C:\Users\polti\Downloads\lexicons\split\JPM_trans.parquet'

# 1. Cargar el Parquet
df = pd.read_parquet(file_path)

# 2. Asegurar que la columna Date es datetime
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# 3. Eliminar filas posteriores al 31‑05‑2024 00:00
cutoff = pd.Timestamp('2024-05-31 00:00:00')
df = df[df['Date'] <= cutoff]

# 4. Sobrescribir el archivo Parquet
df.to_parquet(file_path, index=False)

In [5]:
import pandas as pd
import os

# Ruta del archivo original
input_path = r'C:\Users\polti\Downloads\lexicons\lexicons.parquet'

# Ruta del directorio de salida
output_dir = r'C:\Users\polti\Downloads\lexicons\split'
os.makedirs(output_dir, exist_ok=True)

# Cargar el DataFrame desde el Parquet
df = pd.read_parquet(input_path)

# Filtrar según condiciones
df_filtered = df[(df['AggregationSubtype'] == 'Mean') & (df['DocAgg'] == 'count')]

# Fragmentar y guardar por valores únicos de 'Id'
for unique_id in df_filtered['Id'].unique():
    df_subset = df_filtered[df_filtered['Id'] == unique_id]
    
    # Limpiar el Id para nombre de archivo
    safe_id = "".join(c if c.isalnum() else "_" for c in unique_id)
    
    output_path = os.path.join(output_dir, f'lexicons_{safe_id}.parquet')
    df_subset.to_parquet(output_path, index=False)

In [12]:
import pandas as pd
import numpy as np

# Cargar el Parquet específico de Apple
df_apple = pd.read_parquet(r'C:\Users\polti\Downloads\lexicons\split\lexicons_GB0005405286.parquet')

# Calcular la columna 'sentiment_score' basada en 'UpDownLogRatio'
df_apple['sentiment_score'] = 1 / (1 + np.exp(-df_apple['UpDownLogRatio'])) * 100

# (Opcional) Ver resumen estadístico
print(df_apple['sentiment_score'].describe())

#quedarme solo con las columnas AggDate, UpDownLogRatio y sentiment_score
df_apple = df_apple[['AggDate', 'UpDownLogRatio', 'sentiment_score']]
df_apple.rename(columns={'AggDate': 'datetime'}, inplace=True)

# Guardar el DataFrame actualizado sobrescribiendo el archivo original
df_apple.to_parquet(r'C:\Users\polti\Downloads\lexicons\split\lexicons_HSBC.parquet', index=False)

#crear una copia del parquet en csv
#df_apple.to_csv(r'C:\Users\polti\Downloads\lexicons\split\lexicons_JPM.csv', index=False)


count    4431.000000
mean       48.571470
std         9.508900
min        11.469051
25%        42.739194
50%        50.000000
75%        54.016328
max        84.503778
Name: sentiment_score, dtype: float64
